# DFND: one enclosed region becomes two

On this dumbbell, DFND recovers the principal anchor-level transition: **one
public Void at 1.4 Å, two distinct anchored Voids at 2.2 Å, and no wet/public
objects at 6 Å**. At 1.4 Å the common parent also carries two provisional chamber
motifs and one throat with recoverable source support.

This is a bounded positive result. Global physical-hull decomposition remains
unresolved on all three profiles, and extra native/catalog objects require
separate interpretation. Read [the system notebook](system.ipynb) for the
independent wall, connecting-path, separator and finite-hull proofs.

Run independently from this directory in a fresh Python 3.14 kernel. This
notebook calls the actual public DFND API, reads retained evidence and renders
new observations. It neither overwrites artifacts nor calls original providers
or servers. Long code inputs are expandable on the website.

In [ ]:
import topomt
import ackredit

from pathlib import Path
from html import escape
import hashlib
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import pyunitwizard as puw
import study

CASE = Path.cwd()
ARTIFACTS = CASE / 'artifacts/dumbbell_v1'
fixture = study.read(ARTIFACTS / 'input.json')
policy = study.read(ARTIFACTS / 'protocol.json')
assert hashlib.sha256((ARTIFACTS / 'input.json').read_bytes()).hexdigest() == policy['input_sha256']
centers = np.asarray(puw.get_value(study.quantity(fixture['coordinates'], 'coordinates'), to_unit='angstroms'))
radii = np.asarray(puw.get_value(study.quantity(fixture['atom_radii'], 'atom_radii'), to_unit='angstroms'))
anchors = study.quantity(policy['anchors'], 'anchors')
anchor_values = np.asarray(puw.get_value(anchors, to_unit='angstroms'))
assert centers.shape == (94, 3) and np.allclose(radii, 1.88, rtol=0, atol=1e-14)
assert fixture['source_atom_indices'] == list(range(94))

def table(headers, rows):
    def cells(values, tag):
        return ''.join(f'<{tag}>{escape(str(value))}</{tag}>' for value in values)
    display(HTML('<table><thead><tr>' + cells(headers, 'th') + '</tr></thead><tbody>' +
                 ''.join('<tr>' + cells(row, 'td') + '</tr>' for row in rows) + '</tbody></table>'))

## Keep the physical model and query fixed

The helper loads the PDB topology, restores the 94 full-precision coordinates,
verifies the 1.88 Å atomic radii and original source atom IDs, and executes
`get_topography(method='dfnd')`. The query uses `with_connectors`, `flag_only`,
`dryface`, zero physical residence/permeability tolerances, epsilon/boundary
margin 1e-6 Å and `min_size=0`. Explicit lengths cross the public/native boundary
with the receiver's declared units.

The audit then checks the complete native wet/dry inventory, source supports,
public parent/child relations and copied hierarchy. Original observations and
this execution retain separate producer versions and source identities.

In [ ]:
reader_protocol = study.read(ARTIFACTS / 'reader_protocol.json')
print('Imported TopoMT:', topomt.__file__)
print('Actual producer version:', topomt.__version__)
print('Reviewed scientific source:', reader_protocol['topomt_source_revision'])
refs = study.references(CASE)
results, observations, point_rows, hierarchies = {}, {}, {}, {}
with ackredit.capture('Dumbbell public reader') as credit:
    for probe in (1.4, 2.2, 6.0):
        topo = study.receive(CASE, probe)
        results[probe] = topo
        observations[probe] = study.describe(topo)
        point_rows[probe] = study.inspect_anchors(topo, anchors)
        hierarchies[probe] = study.inspect_hierarchy(topo)
current_credit = credit.attribution.to_dict()
assert current_credit
original = study.read(ARTIFACTS / 'native_observations.json')
assert original['input_sha256'] == policy['input_sha256']

## Follow the named anchors into Topography

Each anchor is located geometrically in the actual tetrahedral mesh. We retain
all containing cells, barycentric coordinates, original atom IDs, native
component/role and corresponding public parent. This avoids selecting a feature
simply because it is largest or has the expected name.

In [ ]:
rows = []
for probe in results:
    for label, point in zip(('left', 'right'), point_rows[probe]):
        rows.append([probe, label, f"{point['physical_clearance_angstroms']:.6f}",
                     ', '.join(str(h['tetrahedron_id']) for h in point['cell_hits']),
                     ', '.join(sorted({h['component_id'] for h in point['cell_hits']})),
                     ', '.join(point['public_parent_ids']) or 'none',
                     ', '.join(point['public_parent_types']) or 'none'])
table(['Probe (Å)', 'Anchor', 'Physical clearance (Å)', 'Native cell',
       'Native component', 'Public parent ID', 'Public type'], rows)
assert point_rows[1.4][0]['wet_component_keys'] == point_rows[1.4][1]['wet_component_keys']
assert point_rows[2.2][0]['wet_component_keys'] != point_rows[2.2][1]['wet_component_keys']
assert not any(p['wet_component_keys'] for p in point_rows[6.0])

At **1.4 Å**, both anchors reach `VOI-1`, consistent with the independently
certified enclosed connection. At **2.2 Å**, they reach distinct public Voids,
consistent with the covered separator while both anchors remain free. At
**6 Å**, the anchors are no longer physically free and their mesh cells are dry;
there are no native wet components or public features anywhere in this query.

Display IDs such as `VOI-1` are local to a result. The same spelling at another
probe does not establish cross-query identity: we compare anchors and original
source supports, not display names alone. The references prove the named-region
transition, rather than exact counts of every physical component.

## Two chamber motifs within one parent

At the fixed 1.4 Å query, the common native parent has two chamber candidates
and one throat candidate. Each anchor's containing tetrahedron belongs to one
of those distinct chamber supports. The throat names those sibling chambers;
all chamber atoms can be recovered from their source-mapped tetrahedra, and
the throat retains its source face atoms and stable motif/support keys.

The same motif dictionaries reach the actual public parent's **`.motifs`**.
Their flags remain `provisional`. They are not certified all-atom subregion
boundaries, and `children_of(VOI-1)` does not manufacture standalone Chamber
objects. Native components and public feature relations are different layers.

In [ ]:
anchor_key = point_rows[1.4][0]['wet_component_keys'][0]
parent = next(c for c in hierarchies[1.4]['components'] if c['component_key'] == anchor_key)
public_parent = next(p for p in hierarchies[1.4]['public'] if p['component_key'] == anchor_key)
assert len(parent['chambers']) == 2 and len(parent['throats']) == 1
assert hierarchies[1.4]['public_motif_keys_match_native']
assert hierarchies[1.4]['standalone_chamber_children_promoted'] is False

def angstroms(native_nm):
    return float(puw.get_value(puw.quantity(native_nm, 'nm'), to_unit='angstroms'))

rows = []
for chamber in parent['chambers']:
    labels = [label for label, point in zip(('left', 'right'), point_rows[1.4])
              if set(h['tetrahedron_id'] for h in point['cell_hits']) & set(chamber['node_ids'])]
    rows.append(['chamber', ', '.join(labels), len(chamber['node_ids']), len(chamber['atom_indices']),
                 f"{angstroms(chamber['separation_radius']):.9f}", ', '.join(chamber['flags'])])
for throat in parent['throats']:
    rows.append(['throat', 'joins the two named chamber supports', 'face support', len(throat['face_atoms']),
                 f"{angstroms(throat['R_gate']):.9f}", ', '.join(throat['flags'])])
table(['Native motif', 'Anchor correspondence', 'Support cells', 'Support atoms',
       'Separation/gate descriptor (Å)', 'Flags'], rows)
print('Public parent:', public_parent['feature_id'])
print('Public hierarchy motif count:', len(public_parent['motif_keys']))
print('Public child feature IDs:', public_parent['public_child_ids'])

The native throat descriptor is **1.537739193 Å**. It is a local graph/capacity
quantity conditional on the selected support, not a certified globally optimal
physical neck radius. The independent axis witness has clearance 1.432602186 Å,
and the separator certifies closure at 2.2 Å. Being consistent with those broad
controls does not validate the exact descriptor value.

At 2.2 Å each anchored Void has no internal chamber/throat candidates in this
profile. Two regions after changing the probe are therefore different from two
motifs inside one region at a fixed probe. No new segmentation is performed by
merely reading the hierarchy.

In [ ]:
from matplotlib.patches import FancyBboxPatch
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), layout='constrained')
for ax in axes:
    ax.set(xlim=(0, 1), ylim=(0, 1)); ax.axis('off')

def box(ax, xy, width, height, text, color):
    ax.add_patch(FancyBboxPatch(xy, width, height, boxstyle='round,pad=0.02',
                              facecolor=color, edgecolor='#476070', linewidth=1.2))
    ax.text(xy[0]+width/2, xy[1]+height/2, text, ha='center', va='center', fontsize=10)

axes[0].set_title('1.4 Å: hierarchy within one anchored Void')
box(axes[0], (.04, .08), .92, .8, '', '#eef3f8')
box(axes[0], (.25, .58), .5, .19, 'One throat candidate
Gate descriptor: 1.537739 Å', '#f6dfb3')
box(axes[0], (.1, .2), .33, .2, 'Left chamber
Provisional motif', '#c4dce8')
box(axes[0], (.57, .2), .33, .2, 'Right chamber
Provisional motif', '#d4e8d6')
axes[0].plot([.36, .26], [.58, .4], color='#476070')
axes[0].plot([.64, .74], [.58, .4], color='#476070')
axes[0].text(.5, .01, 'Motifs ride on the public parent; they are not child features.', ha='center', fontsize=8)
axes[1].set_title('2.2 Å: two distinct anchored Voids')
box(axes[1], (.07, .24), .36, .42, 'Left anchor
VOI-1', '#c4dce8')
box(axes[1], (.57, .24), .36, .42, 'Right anchor
VOI-2', '#d4e8d6')
axes[1].plot([.5, .5], [.12, .8], color='#b04148', ls='--')
axes[1].text(.5, .04, 'The independent covered section excludes anchor communication.', ha='center', fontsize=8)
plt.show()

## Keep additional objects and physical availability visible

The native/public registry contains more than the anchored interior regions.
At 1.4 Å it also reports an OpenConcavity/Mouth; at 2.2 Å it reports several
surface candidates. These are not automatically additional enclosed cavities,
nor are they disproved by a reference that concerns only the two named anchors.
Their full supports and classifications remain in the original observations.

In [ ]:
from collections import Counter
rows = []
for probe, topo in results.items():
    observed = observations[probe]
    saved = next(r for r in original['probes'] if r['probe_angstroms'] == probe)
    assert observed['node_inventory'] == saved['public_observation']['node_inventory']
    types = [f['python_class'] for f in observed['public_features']]
    assert types == [f['python_class'] for f in saved['public_observation']['public_features']]
    physical = topo.dfnd.raw['boundary_connectivity']
    assert physical['status'] == saved['raw']['boundary_connectivity']['status']
    assert physical['n_patches'] is None and physical['reason']
    rows.append([probe, observed['node_inventory']['wet_nodes'], len(observed['wet_components']),
                 ', '.join(f'{name}: {count}' for name, count in Counter(types).items()) or 'none', physical['status'], physical['n_patches'] or 'unknown',
                 physical['reason']])
table(['Probe (Å)', 'Wet nodes', 'Wet components', 'Public catalog counts',
       'Physical hull status', 'Patch count', 'Reason'], rows)

The native finite carrier is the **convex hull H**; the independent enclosure
proof uses a separate **nonconvex barrier P**. A physical result on P does not
supply the count of every strict-free patch on H. The global hull count is
unknown at 1.4 Å because critical-section ordering lies within its numerical
margin ([#184](https://github.com/uibcdf/topomt/issues/184)). At 2.2 and 6 Å,
arrangement work exhausts its budget ([#180](https://github.com/uibcdf/topomt/issues/180)).

At 6 Å the independent complete cover **does** prove zero physical hull patches.
The actual-public zero-count target first fails and remains a strict expected
failure under #180. Empty native/public results already agree with exclusion;
this input does not reproduce the local-candidate survival observed on the tubes
under [#181](https://github.com/uibcdf/topomt/issues/181). A positive control here
does not resolve their earlier source-scoped findings.

## What remains for the joint review

This case supports anchor communication/separation and the transport of an
inspectable provisional hierarchy into Topography. It does not validate all
extra native objects, every motif on other components, exact all-atom chamber
boundaries, physical volume or an optimal neck bottleneck.

The historical claim that a larger probe must find fewer components is not a
valid physical oracle. Shrinking free sets can fragment; the original five-domain
diagnosis at 2.0 Å remains [#188](https://github.com/uibcdf/topomt/issues/188),
`needs_review`. The later [six-case review #187](https://github.com/uibcdf/topomt/issues/187)
will inspect those supports together with existing #180/#181/#184/#76 questions.
No scientific operator was retuned in this delivery.

In [ ]:
from collections import Counter
rows = []
for probe, topo in results.items():
    attribution = topomt.get_attribution(topo)
    assert attribution['status'] == 'recorded'
    rows.append([probe, attribution['producer']['version'], attribution['status'], len(attribution['references'])])
table(['Probe (Å)', 'Actual producer version', 'Attribution status', 'Reference records'], rows)
print('Original native source:', original['source_revision'])
print('Original producer:', original['probes'][0]['topography_attribution']['producer'])
assert isinstance(topomt.cite(results[1.4], format='bibtex'), str)
print('Completed-result bibliography renders without rerunning the analysis.')

The original capture preserves its actual producer version and the hash of the
pre-existing human version-file edit. This reader is separately checked with a
clean committed source tree. References remain detached from the completed
results; reading or rendering them earns no new calculation credit. The earlier
study-serialization correction #185 applies to these new captures and does not
rewrite historical evidence.

The [evidence guide](artifacts/dumbbell_v1/README.md) gives protocols, source
identities, resource receipts and the complete download. All previous study
artifacts/notebooks/ZIPs are preserved. This known input is retrospective
case-guided development, not a blind or original-engine comparison, installed
release/platform qualification or a biological-accuracy claim.